# Phase 1 Notebook: Mention Detection
Reads text archives from data/01_input and writes only to data/10_mention_detection.

## 1) Project Setup
Resolve repository paths and make the source package importable in this notebook session.

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(8):
        if (cur / 'data').exists() and (cur / 'speakermining' / 'src').exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise RuntimeError('Repository root not found.')

ROOT = find_repo_root(Path.cwd())
SRC = ROOT / 'speakermining' / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ROOT

## 2) Input Discovery
Build the list of archive text files that will be parsed into structured episode blocks.

In [ ]:
from process.mention_detection.config import DEFAULT_PDF_TXT_INPUTS, ZDF_ARCHIVE_DIR, PHASE_DIR
from process.text_extraction.text import load_episode_blocks_from_many

input_paths = [ROOT / ZDF_ARCHIVE_DIR / name for name in DEFAULT_PDF_TXT_INPUTS]
input_paths

## 3) Episode And Season Extraction
Parse text blocks into episode records, then derive season-level aggregates from extracted episode metadata.

In [ ]:
from process.mention_detection.episode import extract_episode_and_publication_rows, save_episodes
from process.mention_detection.publications import save_publications
from process.mention_detection.season import extract_season_rows, save_seasons

episode_blocks = load_episode_blocks_from_many(input_paths)
episodes_df, publications_df = extract_episode_and_publication_rows(episode_blocks)
seasons_df = extract_season_rows(episodes_df)

ep_path = save_episodes(episodes_df, ROOT / PHASE_DIR)
pu_path = save_publications(publications_df, ROOT / PHASE_DIR)
se_path = save_seasons(seasons_df, ROOT / PHASE_DIR)

print(f'episodes: {len(episodes_df)} -> {ep_path}')
print(f'publications: {len(publications_df)} -> {pu_path}')
print(f'seasons: {len(seasons_df)} -> {se_path}')

## 4) Mention Extraction
Extract person, institution, and topic mentions from the same episode text blocks.

In [ ]:
from process.mention_detection.guest import extract_person_mentions, save_person_mentions
from process.mention_detection.institution import extract_institution_mentions, save_institution_mentions
from process.mention_detection.topic import extract_topic_mentions, save_topic_mentions

persons_df = extract_person_mentions(episode_blocks, episodes_df)
institutions_df = extract_institution_mentions(episode_blocks, episodes_df)
topics_df = extract_topic_mentions(episode_blocks, episodes_df)

pe_path = save_person_mentions(persons_df, ROOT / PHASE_DIR)
in_path = save_institution_mentions(institutions_df, ROOT / PHASE_DIR)
to_path = save_topic_mentions(topics_df, ROOT / PHASE_DIR)

print(f'persons: {len(persons_df)} -> {pe_path}')
print(f'institutions: {len(institutions_df)} -> {in_path}')
print(f'topics: {len(topics_df)} -> {to_path}')

## 5) Quick Validation
Inspect a few rows from each table to verify shape and extraction quality before continuing.

In [ ]:
seasons_df

In [ ]:
episodes_df

In [ ]:
publications_df

In [ ]:
persons_df

In [ ]:
topics_df

In [ ]:
institutions_df